# Phase 5: Evaluation & export

**Goal:** check that fine-tuning helped, then save (and optionally share) the model.

Assumes **`model`**, **`tokenizer`**, and training from earlier phases exist in your session. Adjust paths and Hub IDs to match your project.

## Imports

`PeftModel` is available if you need explicit PEFT types; merging below uses methods on your trained adapter model.

In [ ]:
from peft import PeftModel

## 5a. Qualitative evaluation

Run inference on a few hand-picked prompts. Use the **same chat template** as in Phase 2 (`add_generation_prompt=True` for generation).

Compare to the **base** model (no LoRA): instruction following, on-topic answers, and hallucinations.

In [ ]:
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "???"}],
    tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 5b. Quantitative evaluation (optional)

Pick a metric for your task: classification → accuracy / F1; summarization → ROUGE; general quality → perplexity on a held-out split.

Install if needed: `pip install evaluate`

In [ ]:
import evaluate

metric = evaluate.load("???")  # e.g. "rouge", "accuracy"

## 5c. Merge LoRA into the base weights

During training, adapter weights are separate. **`merge_and_unload()`** fuses them into a standard model (no PEFT at inference).

**Note:** merging a **4-bit** model in memory is often not supported — reload in float16 first, or **save the adapter only** and load it with `PeftModel.from_pretrained` later.

In [ ]:
merged_model = model.merge_and_unload()

## 5d. Save merged model locally

Creates a folder you can reload with `from_pretrained`, or push with `push_to_hub`.

In [ ]:
merged_model.save_pretrained("./phi3-merged")
tokenizer.save_pretrained("./phi3-merged")

## 5e. (Optional) Push to the Hugging Face Hub

Authenticate first (`huggingface-cli login`). Replace `your-username/...` with your repo id.

In [ ]:
merged_model.push_to_hub("your-username/phi3-mini-finetuned")
tokenizer.push_to_hub("your-username/phi3-mini-finetuned")